# Power from EnergAIzer's measured LUT, not a roofline

Every figure in the other notebooks is stamped `SYNTHETIC`. This one is not.

**Why the swap matters more than 'a better model'.** The analytic backend is an
*ansatz*: `time = max(flops/peak, bytes/bandwidth)`, power blended from the two
utilisation fractions. It gets trends right **by construction** -- which is exactly why
it can never falsify anything. It agrees with the reasoning that produced it, so a
result derived from it is a restatement of that reasoning, not evidence.

EnergAIzer's numbers come from kernels that were **actually run and measured** on an
A100-40GB-PCIe, at 3.1-3.8% MAPE per kernel type. It can disagree with the roofline,
and where it does is the only place there is new information.

| | analytic | EnergAIzer |
|---|---|---|
| source | closed-form roofline | measured lookup tables |
| needs | nothing | the artifact + a ~GB database download |
| can it surprise you? | no | yes -- that is the point |

**No GPU needed.** The measurements were taken on an A100 once; this is a lookup.

The download in section 3 is the slow step. Everything after it is fast.

## 1 - The simulator

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'

if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=True)

if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'pytest'], check=True)

import dynshape
print('dynshape', dynshape.__version__)

## 2 - The EnergAIzer artifact

The code, and the dependencies its estimator needs. `gee` imports `torch`, `cvxpy`,
`scikit-learn`, `scipy` and `opt_einsum` -- Colab has most of them already.

In [ ]:
from dynshape.energaizer import clone_artifact, locate_artifact, lut_status

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml', 'scipy', 'scikit-learn', 'cvxpy', 'opt-einsum',
                'tqdm', 'gdown'], check=True)

ART = clone_artifact('/content/energaizer')
print()
print(locate_artifact(ART).describe())

## 3 - The measured database

**This is the piece that does not ship.** `database/` in the artifact holds the harness
that *collects* a database, not a database. The measured CSVs are a separate Google
Drive download, which is the entire reason every figure so far has said SYNTHETIC.

Expect a few minutes. If the Drive link rate-limits, section 3b has the manual route.

In [ ]:
from dynshape.energaizer import download_lut

lut_dir = download_lut(ART)

paths = locate_artifact(ART)
st = lut_status(paths)
print()
print(f"complete : {st['complete']}")
print(f"tables   : {len(st['present'])} of {len(st['wanted'])}, {st['total_mb']:.0f} MB")
for name in st['present']:
    size = os.path.getsize(os.path.join(st['lut_dir'], name)) / 1024**2
    print(f"    {size:7.1f} MB  {name}")
if st['missing']:
    print('MISSING:')
    for name in st['missing']:
        print('   ', name)

**Why the completeness check is not pedantry.** A *partially* extracted database builds
an estimator that works for most op types and raises on one. With `skip_unsupported`
on -- the default, because it keeps a run alive -- those kernels are silently dropped,
and the trace comes back simply cheaper than reality with no error anywhere.
`build_gee_predictor` refuses to start rather than let that happen.

### 3b - If the download failed

Download [precollected_database.tar.gz](https://drive.google.com/file/d/1krvqRFDnaqrJUT06V2psIua0wQr6ETAE/view)
by hand, upload it to `/content/energaizer/.../database/data`, and run:

```python
!cd /content/energaizer/energaizer-ispass26-artifact-main/database/data && tar -xzf precollected_database.tar.gz
```

## 4 - Build the measured predictor

`build_gee_predictor` **raises** if anything is missing, unlike `build_predictor` which
degrades to the roofline. Silent fallback is right for a demo and wrong here: if you
asked for EnergAIzer you want to know when you did not get it, not to read SYNTHETIC
numbers under a heading that says measured.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dynshape import ShapeRewriter, CachedPredictor, AnalyticBackend
from dynshape.energaizer import build_gee_predictor

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

rw   = ShapeRewriter.from_dir('templates/gpt2')
gee  = build_gee_predictor(ART)                       # measured
ana  = CachedPredictor(backend=AnalyticBackend(), freq=900)   # roofline, for contrast

print()
print('measured backend :', gee.backend.name)
print('is_measured_model:', gee.backend.is_measured_model)
print('frequency        :', gee.freq, 'MHz  (the A100 tables are measured at 900)')

## 4b - Two details from the authors' own demo

Both were checked against `EnergAIzer_Colab_Demo.ipynb` rather than guessed, and both
are easy to get wrong in a way that fails quietly.

**1. The LUT config references tables by bare filename.** They are resolved against one
`lut_folder_abs_path`, so a CSV that extracted a directory deeper is invisible. The
failure mode is not `FileNotFoundError` -- it is a working estimator that raises on one
op type, which `skip_unsupported` then drops, handing back a trace that is simply
cheaper than reality. `flatten_lut` (called automatically) symlinks everything up.

**2. Idle power is measured, and it is not 47.0 W.** Every figure so far has used a
round 47.0 read off a config file. The measured table says **47.35 W at 900 MHz** --
and, more importantly, idle is *not flat*: 44.5 W at 210 MHz, 67.3 W at 1410 MHz. A run
at boost clock pays half again as much for doing nothing.

In [ ]:
from dynshape.energaizer import flatten_lut, idle_power_table, measured_idle_power_w
from dynshape.simulate import IDLE_W

print('CSVs directly in database/data:',
      len([f for f in os.listdir(flatten_lut(ART)) if f.endswith('.csv')]))
print()

idle = idle_power_table(ART)
print(f'idle power we have been assuming : {IDLE_W:.2f} W')
print(f'idle power measured at 900 MHz   : {measured_idle_power_w(ART, 900):.2f} W')
print()
for f in (210, 510, 900, 1020, 1200, 1410):
    print(f'  {f:5d} MHz   {idle[f]:6.2f} W')
print()
print(f'idle at boost is {idle[1410]/idle[210]:.2f}x idle at minimum clock -- which is the')
print('mechanism behind the U-shaped energy-vs-clock curve: raising the clock finishes')
print('the work sooner and pays this floor for less time, until voltage starts climbing')
print('and both the V^2 term and this number rise faster than the runtime shrinks.')

MEASURED_IDLE_W = measured_idle_power_w(ART, 900)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
fs = sorted(idle)
ax.plot(fs, [idle[f] for f in fs], lw=1.4, color='#8e44ad')
ax.axvline(900, ls='--', color='gray', lw=1)
ax.axhline(IDLE_W, ls=':', color='#c0392b', lw=1,
           label=f'the {IDLE_W:.0f} W constant used so far')
ax.annotate('LUT measured here', xy=(900, idle[900]), xytext=(700, 60),
            arrowprops=dict(arrowstyle='->', lw=0.8), fontsize=8)
ax.set_xlabel('core clock (MHz)'); ax.set_ylabel('idle power (W)')
ax.set_title('A100-40GB-PCIe idle power is measured, and it is not flat')
ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 5 - A first lookup

One transformer block's worth of kernels, both ways.

In [ ]:
kernels = rw.expand(batch=8, seqlen=512, mode='prefill')
print(f'{len(kernels)} kernels for b8 s512 prefill\n')

rows = []
for q, op in kernels[:14]:
    t_g, p_g, e_g = gee.predict(q, op)
    t_a, p_a, e_a = ana.predict(q, op)
    rows.append({'op': ' '.join(op),
                 'shape': str({k: v for k, v in q.items()
                               if k in ('batch','dim','dimM','dimN','dimK')}),
                 'LUT ms': t_g, 'LUT W': p_g,
                 'roofline ms': t_a, 'roofline W': p_a,
                 'time ratio': t_g / t_a, 'power ratio': p_g / p_a})
display(pd.DataFrame(rows).round(4))

## 6 - Does EnergAIzer's own arithmetic close?

`lookup` returns time, power and energy from related but **distinct** prediction paths,
so `energy == power x time` is not guaranteed. Downstream code assumes it does -- an
iteration's average power is its summed energy over its summed time.

Rather than silently pick one, the backend measures the disagreement. If it is tiny the
question is moot; if it is not, you need to decide which number you trust, and
`reconcile=` says so explicitly instead of a comment hiding the choice.

In [ ]:
gap = gee.stats()['max_energy_vs_power_x_time']
print(f'largest |E - P*t| / E seen so far: {gap:.3%}  over {gee.stats()["lut_lookups"]} lookups')
print()
if gap < 0.01:
    print('Under 1% -- the three predictions are mutually consistent, so the choice')
    print('of reconciliation policy does not matter and the default (report as')
    print('measured, change nothing) is the right one.')
else:
    print('Large enough to matter. Every aggregate in this project derives average')
    print('power from energy over time, so per-kernel power and those aggregates')
    print('will disagree by roughly this much. Rebuild with reconcile="energy" to')
    print('force consistency, and say so wherever the numbers are reported.')

## 7 - Where the roofline was wrong

The interesting cell. Same shapes, both models, across the whole prefill/decode grid.

A roofline cannot be wrong about *direction* -- bigger is slower, compute-bound is
hotter -- because those are baked into its form. It can be badly wrong about
**magnitude**, and specifically about the regimes the paper flags: kernels under 0.05 ms
and small-`M` GEMMs, which is exactly where decode lives.

In [ ]:
# NOTE: every (batch, seqlen, mode) here is ~242 lookups, most of them distinct,
# and a cold measured lookup is 50-300 ms. This grid is deliberately small --
# widening it is the easiest way to turn this cell into a coffee break.
rows = []
for mode in ('prefill', 'decode'):
    for b in (1, 8):
        for s in (128, 512, 2048):
            ks = rw.expand(batch=b, seqlen=s, mode=mode)
            tg = sum(gee.predict(q, op)[0] for q, op in ks)
            eg = sum(gee.predict(q, op)[2] for q, op in ks)
            ta = sum(ana.predict(q, op)[0] for q, op in ks)
            ea = sum(ana.predict(q, op)[2] for q, op in ks)
            rows.append({'mode': mode, 'batch': b, 'seqlen': s,
                         'LUT ms': tg, 'roofline ms': ta, 'time ratio': tg/ta,
                         'LUT W': eg/(tg/1000), 'roofline W': ea/(ta/1000),
                         'power ratio': (eg/tg)/(ea/ta)})
grid = pd.DataFrame(rows)
display(grid.round(3))

print()
for mode in ('prefill', 'decode'):
    g = grid[grid['mode'] == mode]
    print(f"{mode:8s} time ratio LUT/roofline: {g['time ratio'].min():.2f} to "
          f"{g['time ratio'].max():.2f}   (median {g['time ratio'].median():.2f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for mode, c, m in (('prefill', '#e67e22', 'o'), ('decode', '#1f4e79', 's')):
    g = grid[grid['mode'] == mode]
    ax[0].scatter(g['roofline ms'], g['LUT ms'], c=c, marker=m, s=60,
                  alpha=0.8, label=mode)
    ax[1].scatter(g['roofline W'], g['LUT W'], c=c, marker=m, s=60,
                  alpha=0.8, label=mode)

lo = min(grid['roofline ms'].min(), grid['LUT ms'].min())
hi = max(grid['roofline ms'].max(), grid['LUT ms'].max())
ax[0].plot([lo, hi], [lo, hi], 'k--', lw=0.8, label='agreement')
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].set_xlabel('roofline (ms)'); ax[0].set_ylabel('EnergAIzer LUT (ms)')
ax[0].set_title('Latency: where the ansatz was wrong')

lo = min(grid['roofline W'].min(), grid['LUT W'].min())
hi = max(grid['roofline W'].max(), grid['LUT W'].max())
ax[1].plot([lo, hi], [lo, hi], 'k--', lw=0.8, label='agreement')
ax[1].set_xlabel('roofline (W)'); ax[1].set_ylabel('EnergAIzer LUT (W)')
ax[1].set_title('Average power')
for a in ax:
    a.legend(fontsize=8); a.grid(alpha=0.25, which='both')
plt.tight_layout(); plt.show()

print('Points off the dashed line are places the roofline was guessing. Anything')
print('systematically off in ONE phase matters most, because that is a bias the')
print('scheduler will then amplify -- decode iterations outnumber prefill ones by')
print('an order of magnitude, so a decode-side error compounds across the run.')

### Per op type

Which kernel families the roofline handles and which it does not. One shape only --
484 more lookups, and at this point most of them are cache hits from the grid above.

In [ ]:
rows = []
for mode in ('prefill', 'decode'):
    for q, op in rw.expand(batch=8, seqlen=1024, mode=mode):
        tg, pg, _ = gee.predict(q, op)
        ta, pa, _ = ana.predict(q, op)
        rows.append({'mode': mode, 'op': ' '.join(op),
                     'LUT ms': tg, 'roofline ms': ta,
                     'time ratio': tg/ta, 'power ratio': pg/pa})
per_op = pd.DataFrame(rows).groupby(['mode', 'op']).agg(
    kernels=('LUT ms', 'size'),
    LUT_ms_total=('LUT ms', 'sum'),
    roofline_ms_total=('roofline ms', 'sum'),
    median_time_ratio=('time ratio', 'median'),
    median_power_ratio=('power ratio', 'median'))
display(per_op.round(3))

## 8 - The full serving engine, on measured power

Same L0 + L1 + mixed batching as the other notebook. Only the predictor changed.

### First, how long will this take?

**Do not skip the calibration cell.** A measured lookup is not a table read: EnergAIzer
re-solves a small quadratic program per distinct shape -- once for time, again for
power -- via `gee/optimization_utils/cvxpy_qp.py`. That is **50-300 ms per shape**
against roughly a microsecond for the roofline.

A serving run produces a few thousand distinct shapes (a 400-request run of this
traffic reported 11,508), so the run cost is

```
minutes  =  seconds_per_lookup  x  distinct_shapes  /  60
```

which lands anywhere between two minutes and two hours depending on the machine. That
is not a factor to discover by waiting, so the next cell measures the rate on ~30 cold
lookups and prints the projection **before** committing to the run.

Cache *hits* are free by comparison -- a dict lookup against a QP solve -- so the
distinct-shape count is the entire cost model. This is what the design doc meant by
"caching is mandatory, not an optimisation".

In [ ]:
import time
from dynshape import (TrafficConfig, generate_traffic, traffic_summary,
                      EngineConfig, SchedulerConfig, run_engine, reset_ids)
from dynshape.energaizer import (benchmark_lookup, estimate_distinct_shapes,
                                 project_run_minutes)
from dynshape.engine_plot import plot_engine_dashboard

N_REQUESTS = 60      # raise this once you know what it costs -- see below

bench = benchmark_lookup(gee, rw.expand(batch=3, seqlen=377, mode='prefill'), n=30)
shapes = estimate_distinct_shapes(N_REQUESTS)
projected = project_run_minutes(bench['seconds_per_lookup'], shapes)

print(f"measured   {1000*bench['seconds_per_lookup']:6.1f} ms per cold lookup "
      f"({bench['lookups_per_second']:.1f}/s)")
print(f"expected   ~{shapes:,} distinct shapes for {N_REQUESTS} requests")
print(f"projected  ~{projected:.0f} minutes for the measured run")
print()
print(f"  at 250 requests that would be ~{project_run_minutes(bench['seconds_per_lookup'], estimate_distinct_shapes(250)):.0f} minutes")
print(f"  at 400 requests, ~{project_run_minutes(bench['seconds_per_lookup'], estimate_distinct_shapes(400)):.0f} minutes")
print()
print('If that is too slow, in order of preference:')
print('  1. lower N_REQUESTS -- cost is linear in it')
print('  2. rebuild with use_precomputed_coeff=True (next cell): reads fitted')
print('     coefficients from a table instead of re-solving, 10-100x faster,')
print('     falling back to the full solve where no entry exists')
print('  3. bucket decode contexts harder -- every distinct KV length is a shape')

### The run

Set `FAST = True` to use precomputed coefficients. It reads the fitted coefficients
from a table instead of re-solving the QP per shape, and falls back to the full solve
wherever no entry exists -- so it degrades in accuracy only where it has to.

It is **off by default** because that is what the artifact's own end-to-end runs use,
and a speed knob that quietly changes numbers should be something you turn on
deliberately. The cell after this one measures what it actually costs you.

In [ ]:
FAST = False          # True -> precomputed coefficients, 10-100x faster

if FAST:
    gee = build_gee_predictor(ART, use_precomputed_coeff=True, verbose=False)
    print('rebuilt with precomputed coefficients')

def make_traffic(seed=0, n=N_REQUESTS):
    reset_ids()
    return generate_traffic(TrafficConfig(
        interval='gamma', qps=100.0, cv=2.0,
        length='zipf', min_tokens=64, max_tokens=2048, theta=0.85,
        prefill_to_decode_ratio=4.0, num_requests=n, seed=seed))

# Idle power from the measured table, not the 47.0 constant. It is the floor
# under every gap and every idle stretch, so at low duty cycle it is most of
# the wall-clock average.
cfg = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                              block_size=16, max_tokens=4096),
    fuse_linear=True, record_kernels_until_ms=40.0,
    idle_w=MEASURED_IDLE_W)

t0 = time.time()
trace_gee = run_engine(make_traffic(), rw, gee, cfg, progress_every=100)
gee_seconds = time.time() - t0
print(f'\nmeasured run: {gee_seconds:.0f} s wall clock, '
      f"{trace_gee.summary()['distinct_shapes']:,} distinct shapes")

t0 = time.time()
trace_ana = run_engine(make_traffic(), rw, ana, cfg)
print(f'roofline run: {time.time() - t0:.0f} s wall clock')
print()
print(f'both runs use the measured idle floor of {MEASURED_IDLE_W:.2f} W, so the only')
print('difference between them is how the busy kernels are priced.')

### What the fast path costs in accuracy

Before trusting `FAST = True` for anything you report, measure it rather than assuming.
Same shapes, both settings, on a sample small enough to be quick.

In [ ]:
from dynshape import CachedPredictor, GeeBackend

# Reuse the estimator already built -- loading the LUT again costs minutes and
# hundreds of MB for no benefit.
est = gee.backend.estimator
exact = CachedPredictor(backend=GeeBackend(est, use_precomputed_coeff=False), freq=900)
fast  = CachedPredictor(backend=GeeBackend(est, use_precomputed_coeff=True), freq=900)

sample = rw.expand(batch=4, seqlen=512, mode='prefill')[:60]
rows = []
for q, op in sample:
    te, pe, ee = exact.predict(q, op)
    tf, pf, ef = fast.predict(q, op)
    rows.append({'op': ' '.join(op), 'exact ms': te, 'fast ms': tf,
                 'time ratio': tf/te if te else np.nan,
                 'exact W': pe, 'fast W': pf,
                 'power ratio': pf/pe if pe else np.nan})
acc = pd.DataFrame(rows)
display(acc.groupby('op')[['time ratio', 'power ratio']]
        .agg(['count', 'median', 'min', 'max']).round(4))

t_err = 100 * (acc['time ratio'] - 1).abs().median()
p_err = 100 * (acc['power ratio'] - 1).abs().median()
print(f'median disagreement: {t_err:.2f}% on time, {p_err:.2f}% on power')
print()
if max(t_err, p_err) < 1.0:
    print('Under a percent -- well inside the 3.1-3.8% MAPE the model claims against')
    print('hardware, so the fast path is not what limits accuracy here.')
else:
    print('Large enough to matter. Use the exact path for anything you report, and')
    print('the fast path only for exploring.')

In [ ]:
keys = ('iterations', 'mixed_fraction', 'wall_time_s', 'busy_time_s',
        'duty_cycle', 'total_energy_j', 'avg_power_w_wallclock',
        'avg_power_w_busy', 'peak_iteration_power_w',
        'energy_per_output_token_mj', 'output_tokens_per_s',
        'ttft_p50_s', 'ttft_p99_s', 'itl_p50_ms', 'itl_p99_ms')
cmp = pd.DataFrame({'EnergAIzer LUT': [trace_gee.summary()[k] for k in keys],
                    'roofline (SYNTHETIC)': [trace_ana.summary()[k] for k in keys]},
                   index=list(keys))
cmp['ratio'] = cmp['EnergAIzer LUT'] / cmp['roofline (SYNTHETIC)']
display(cmp.round(4))

**Read the `iterations` and `mixed_fraction` rows first.** They should be *identical*.
The scheduler is deterministic given the traffic, and the traffic is seeded -- so the
same requests were admitted in the same order and batched the same way in both runs.

That is the check that makes everything else interpretable: **only the prices changed,
not the decisions.** If those rows differ, the predictor is feeding back into the
schedule through the clock, and the two traces are not comparable.

Wait -- they *can* differ, and that is worth understanding. EnergAIzer is the timing
authority: iteration duration is the sum of kernel times, so a different predictor
moves the clock, which moves which requests have arrived by the time each batch is
formed. Under bursty arrivals that changes the batching. This is not a bug; it is the
whole reason the timing authority has to be a single source.

In [ ]:
fig = plot_engine_dashboard(trace_gee, zoom_ms=40.0)
plt.show()

Check the stamp in the bottom-left corner. It should no longer say SYNTHETIC.

### The trace, in PowerTrace-Sim's figure style

Same conventions as `power-test/plot_best_rate_traces.py`, so a figure from here can
sit next to one of theirs without the rendering getting in the way. Four choices in
that figure are deliberate, and all four are worth keeping:

- **one-second means** -- an event trace has variable-width steps and cannot be
  overlaid with anything; a fixed grid is what a meter records and the only thing two
  runs can share.
- **alpha gradient along each line** -- opacity rises with elapsed time, so where the
  two lines sit on top of each other you can still tell them apart *and* see which way
  time runs. On a dense trace this does more work than colour alone.
- **4.4 x 2.5 inches** -- the figure is drawn at the size it will be read at, so the
  line weights are honest. A trace that looks clean at 13 inches wide and turns into a
  solid block in a paper column was never really legible.
- **legend above the axes** -- it can never cover the trace.

**One thing does not port, and it matters.** In their figure the black line is an NVML
capture from real hardware. Nothing here is that. The closest available reference is
EnergAIzer's measured LUT, so that is what goes in black -- and the label says so
rather than borrowing the word *measured*.

In [ ]:
from dynshape.engine_plot import plot_power_trace_paper, trace_agreement

DT_S = 0.05        # the run is a few seconds long, so 1 s would be ~5 points

ax = plot_power_trace_paper(
    [('EnergAIzer LUT', trace_gee), ('Roofline (SYNTHETIC)', trace_ana)],
    dt_s=DT_S)
plt.show()

print(f'Resampled at {DT_S*1000:.0f} ms. FSTS uses 1 s over a 600 s trace; this run is')
print(f'{trace_gee.total_time_ms/1000:.1f} s, so the aperture is scaled to keep a')
print('comparable number of points on the page.')

### And their agreement metrics

Ported from `feature-test/evaluation_core.py::trace_metrics`, because *which* metrics
they chose is the interesting part: **energy error alone is not enough.** Two traces
can carry identical total energy while one is a flat line and the other swings between
idle and peak -- and for anything that sizes a breaker those are entirely different
traces.

| metric | the question it asks |
|---|---|
| `energy_error_pct` | do the totals agree? |
| `mean_bias_pct` | signed, so systematic over- or under-prediction shows |
| `nrmse_range` | point-by-point error against the dynamic range |
| `acf_r2` | does it wobble on the same **timescales**? |
| `ks_agreement` | does it visit the same power **levels**? |

Read here as *how far the roofline is from the measured tables* -- not as an accuracy
claim for either, since neither line is hardware.

In [ ]:
agree = trace_agreement(trace_gee, trace_ana, dt_s=DT_S)
for k, v in agree.items():
    print(f'  {k:20s} {v:12.4f}' if isinstance(v, float) else f'  {k:20s} {v:>12}')

print()
print('How to read these:')
print(f"  the roofline is {agree['mean_bias_pct']:+.1f}% off the measured tables on mean power,")
print(f"  and {agree['energy_error_pct']:.1f}% off on total energy.")
if agree['ks_agreement'] < 0.8:
    print('  KS agreement is low: the two traces visit DIFFERENT power levels, which')
    print('  a mean or an energy total would have hidden entirely.')
if agree['acf_r2'] < 0.5:
    print('  acf_r2 is low: they also wobble on different timescales -- the shape of')
    print('  the trace differs, not just its height.')

In [ ]:
# Their layout for a sweep: one small panel per condition, shared y-axis.
fig, axes = plt.subplots(1, 3, figsize=(13.2, 2.5), sharey=True)
tg, pg = trace_gee.resample(dt_ms=DT_S*1000)
ta, pa = trace_ana.resample(dt_ms=DT_S*1000)
ymax = 1.05 * max(pg.max(), pa.max())

for ax, (name, tr) in zip(axes, [('EnergAIzer LUT', trace_gee),
                                 ('Roofline (SYNTHETIC)', trace_ana)]):
    plot_power_trace_paper([(name, tr)], dt_s=DT_S, ax=ax, ylim=ymax)

plot_power_trace_paper([('EnergAIzer LUT', trace_gee),
                        ('Roofline (SYNTHETIC)', trace_ana)],
                       dt_s=DT_S, ax=axes[2], ylim=ymax)
for ax in axes[1:]:
    ax.set_ylabel('')
plt.tight_layout(pad=0.4, rect=(0, 0, 1, 0.88))
plt.show()

## 9 - Fixed-rate power, measured

The same resampling as before, now on numbers a meter could actually have produced --
which is the point at which comparing against an NVML capture becomes meaningful.

In [ ]:
from dynshape.engine_plot import plot_resampled_power, plot_work_vector

fig, axes = plt.subplots(3, 1, figsize=(13, 10))
plot_resampled_power(trace_gee, dt_ms=1.0, smooth_tau_ms=5.0, ax=axes[0], max_ms=600)
axes[0].set_title('EnergAIzer LUT -- ' + axes[0].get_title())
plot_resampled_power(trace_ana, dt_ms=1.0, smooth_tau_ms=5.0, ax=axes[1], max_ms=600)
axes[1].set_title('roofline (SYNTHETIC) -- ' + axes[1].get_title())

tg, pg = trace_gee.resample(dt_ms=1.0, smooth_tau_ms=5.0)
ta, pa = trace_ana.resample(dt_ms=1.0, smooth_tau_ms=5.0)
axes[2].plot(tg, pg, lw=1.2, color='#c0392b', label='EnergAIzer LUT')
axes[2].plot(ta, pa, lw=1.2, color='#888888', label='roofline')
axes[2].set_xlim(0, 600); axes[2].set_xlabel('wall clock (ms)')
axes[2].set_ylabel('W'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.25)
axes[2].set_title('Overlaid, both through a 5 ms board response')
plt.tight_layout(); plt.show()

## 10 - The work vector is unchanged, and that is the point

FLOPs and bytes come from the **shapes**, not from a power model, so swapping the
predictor cannot move them. If the scheduler ran the same way, the work vectors must be
identical -- and any difference in watts is then unambiguously the predictor's.

That is the separation the work vector exists for: with only watts reported, a
disagreement between two runs could be a scheduler bug or a model difference, and you
could not tell which.

In [ ]:
wg, wa = trace_gee.work_totals(), trace_ana.work_totals()
work = pd.DataFrame({'EnergAIzer run': wg, 'roofline run': wa})
work['ratio'] = work['EnergAIzer run'] / work['roofline run']
display(work.round(4))

same_schedule = (trace_gee.summary()['iterations'] == trace_ana.summary()['iterations'])
print(f'same number of iterations: {same_schedule}')
if same_schedule:
    print('-> the work is identical, so every difference above is the power model.')
else:
    print('-> the schedules diverged (EnergAIzer is the timing authority, so a')
    print('   different clock reorders arrivals). The work differs a little as a')
    print('   result, and the two traces are close cousins rather than twins.')

fig, ax = plt.subplots(figsize=(13, 3.4))
plot_work_vector(trace_gee, ax=ax, max_ms=600)
plt.tight_layout(); plt.show()

## 11 - Export

In [ ]:
idf, rdf, sdf = trace_gee.to_dataframes()
t_grid, p_grid = trace_gee.resample(dt_ms=1.0)
_, p_smooth = trace_gee.resample(dt_ms=1.0, smooth_tau_ms=5.0)
pdf = pd.DataFrame({'t_ms': t_grid, 'power_w': p_grid,
                    'power_w_board_5ms': p_smooth})

for name, df in (('measured_iterations', idf), ('measured_requests', rdf),
                 ('measured_kernels', sdf), ('measured_power_1ms', pdf),
                 ('lut_vs_roofline_grid', grid)):
    df.to_csv(f'{name}.csv', index=False)
    print(f'{name}.csv', df.shape)

try:
    from google.colab import files
    for name in ('measured_iterations', 'measured_power_1ms', 'lut_vs_roofline_grid'):
        files.download(f'{name}.csv')
except Exception:
    print('(not on Colab)')

## What is measured now, and what still is not

**Measured:** per-kernel latency, power and energy, from tables collected on a real
A100-40GB-PCIe at 900 MHz, at 3.1-3.8% MAPE per kernel type. Also the kernel shapes
themselves -- prefill from 25 shipped traces, decode from four traced on hardware.

**Still not measured, and these are what would change a number:**

1. **Additivity.** Kernel costs are summed with no overlap, no memory-system
   contention, no cache carry-over. Every kernel here is priced as if it ran alone.
   The LUT makes each term accurate; it says nothing about whether summing them is.
2. **HuggingFace kernels, vLLM schedule.** The scheduler assumes paged attention; the
   shapes come from an eager HF trace. Expect systematic over-prediction of decode.
3. **The idle and gap constants** are still 47 W and 0.05 ms from a config file, not
   from the LUT.
4. **Decode is the least *accurate* region of the LUT**, not the least covered -- the
   database is deliberately skewed small-`M` (57.9% of bf16 rows have `M` <= 64). The
   artifact's own Fig. 2a shows ~96 W of spread at identical latency there. Decode
   iterations outnumber prefill ones by an order of magnitude, so that spread is where
   an uncertainty layer would pay for itself first.
5. **One replica, one GPU.** No routing, no facility aggregation.